In [ ]:
from __future__ import annotations

import argparse
import atexit
import itertools
import json
import math
import os
import time
import warnings
from collections import deque
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

TSRD_REPO = "alan-turing-institute/turing-synthetic-radar-dataset"


@dataclass
class Config:
    # Environment
    num_bins: int = 10
    k_observed: int = 3
    snr_db: float = 10.0
    freq_range_ghz: Tuple[float, float] = (2.0, 6.0)
    frame_window_us: float = 10.0
    episode_length: int = 200
    detection_threshold: float = 0.5
    false_alarm_penalty: float = 2.0
    # ---- state ----
    seq_len: int = 8
    use_ekf: bool = False
    ekf_pri_min: float = 2.0
    ekf_pri_max: float = 30.0
    # ---- DQN ----
    lr: float = 1e-4
    gamma: float = 0.90
    batch_size: int = 32
    buffer_capacity: int = 20_000
    target_update_freq: int = 300
    hidden_dim: int = 64
    head_dim: int = 128
    gru_dropout: float = 0.0
    # ---- exploration ----
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_steps: int = 2_000
    guided_fraction: float = 0.5
    warmup_steps: int = 1_000
    learn_every: int = 1
    # ---- reward shaping ----
    anomaly_weight: float = 5.0
    surprise_decay: float = 0.999
    redundancy_weight: float = 1.0
    redundancy_grace: int = 2
    redundancy_cap: int = 5


ENV_FIELDS = ("num_bins", "k_observed", "freq_range_ghz", "frame_window_us", "episode_length",
              "detection_threshold", "false_alarm_penalty")

#Data source 

PDW_COLUMNS = ["toa_us", "freq_ghz", "pw_us", "aoa_deg", "amplitude_db", "emitter_id"]


class PdwSource:
    """Yields one contiguous window of pulses per call (DataFrame with PDW_COLUMNS, sorted by
    toa_us, toa re-based to 0, frequency in GHz, ALL frequencies - the env applies the band)."""

    def sample_window(self, span_us: float, rng: np.random.Generator) -> pd.DataFrame:
        raise NotImplementedError

    def close(self) -> None:
        pass

    def stats(self) -> str:
        return ""


class _CountingFile:
    """Proxy around an fsspec file that counts bytes h5py asks for."""

    def __init__(self, raw, owner: "H5StreamSource"):
        self._raw, self._owner = raw, owner

    def readinto(self, b):
        n = self._raw.readinto(b)
        self._owner.bytes_requested += n or 0
        return n

    def read(self, *a, **k):
        d = self._raw.read(*a, **k)
        self._owner.bytes_requested += len(d)
        return d

    def __getattr__(self, name):
        return getattr(self._raw, name)


class H5StreamSource(PdwSource):
    def __init__(self, fs, base_dir: str, mode: str = "stare", subset: str = "train",
                 episodes_per_file: int = 4, initial_rows: int = 8_000,
                 max_rows_per_read: int = 500_000, block_size_mb: float = 0.5,
                 cache_type: str = "blockcache", max_retries: int = 4, freq_unit: str = "auto",
                 aoa_unit: str = "deg", verbose: bool = True):
        try:
            import h5py  # noqa: F401
        except ImportError as e:
            raise ImportError("h5py is required:  pip install h5py") from e
        self.fs, self.mode, self.subset = fs, mode, subset
        self.dir = f"{base_dir.rstrip('/')}/{mode}/{subset}_{mode}"
        self.episodes_per_file, self.initial_rows = episodes_per_file, initial_rows
        self.max_rows_per_read, self.block_size_mb = max_rows_per_read, block_size_mb
        self.cache_type, self.max_retries = cache_type, max_retries
        self.freq_unit, self.aoa_unit, self.verbose = freq_unit, aoa_unit, verbose
        self.bytes_requested = 0
        self.transport_bytes = 0
        self.files_opened = 0
        self._h5 = self._raw = self._layout = None
        self._episodes_left = 0
        atexit.register(self.close)
        self.files = self._list_files()
        if not self.files:
            raise FileNotFoundError(
                f"No .h5 files found under '{self.dir}'. For Hugging Face check that you accepted the "
                f"dataset terms and that your token is valid; for --source local check --local-dir.")
        if verbose:
            print(f"  [source] {len(self.files)} pulse-train files under {self.dir}")

    def _list_files(self) -> List[str]:
        paths = [p for p in self.fs.ls(self.dir, detail=False) if str(p).endswith(".h5")]

        def key(p):
            stem = os.path.basename(str(p))[:-3]
            try:
                return (0, int(stem.split("_")[-1]))
            except ValueError:
                return (1, stem)
        return sorted(paths, key=key)

    def _transport_now(self) -> int:
        cache = getattr(getattr(self._raw, "_raw", None), "cache", None)
        return int(getattr(cache, "total_requested_bytes", 0) or 0)

    def _close(self):
        self.transport_bytes += self._transport_now()
        for obj in (self._h5, self._raw):
            try:
                if obj is not None:
                    obj.close()
            except Exception:
                pass
        self._h5 = self._raw = self._layout = None

    def close(self):
        self._close()

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()

    @staticmethod
    def _read_layout(h5) -> Dict[str, int]:
        cols = {"toa": 0, "freq": 1, "pw": 2, "aoa": 3, "amp": 4}
        try:
            names = [x.decode() if isinstance(x, bytes) else str(x) for x in h5["metadata"]["feature_names"][()]]
            found = {}
            for i, nm in enumerate(names):
                l = nm.lower()
                if "toa" in l or "time" in l:
                    found["toa"] = i
                elif "freq" in l or l == "cf" or "centre" in l or "center" in l:
                    found["freq"] = i
                elif "pw" in l or "width" in l:
                    found["pw"] = i
                elif "aoa" in l or "angle" in l:
                    found["aoa"] = i
                elif "amp" in l or "power" in l:
                    found["amp"] = i
            if len(found) == 5 and len(set(found.values())) == 5:
                cols = found
        except Exception:
            pass
        return cols

    def _open(self, rng: np.random.Generator):
        import h5py
        self._close()
        path = self.files[int(rng.integers(len(self.files)))]
        kwargs = dict(block_size=int(self.block_size_mb * 2 ** 20), cache_type=self.cache_type)
        try:
            raw = self.fs.open(path, "rb", **kwargs)
        except TypeError:
            raw = self.fs.open(path, "rb")
        self._raw = _CountingFile(raw, self)
        self._h5 = h5py.File(self._raw, "r")
        self._layout = self._read_layout(self._h5)
        self._episodes_left = self.episodes_per_file
        self.files_opened += 1
        if self.verbose:
            size = getattr(raw, "size", None)
            print(f"  [source] opened {os.path.basename(str(path))}"
                  + (f" ({size / 2 ** 20:.0f} MB remote)" if size else "")
                  + f", {self._h5['data'].shape[0]:,} pulses")

    def sample_window(self, span_us, rng):
        last_err = None
        for attempt in range(self.max_retries):
            try:
                if self._h5 is None or self._episodes_left <= 0:
                    self._open(rng)
                df = self._sample_once(span_us, rng)
                self._episodes_left -= 1
                return df
            except Exception as e:
                last_err = e
                warnings.warn(f"stream read failed ({type(e).__name__}: {e}); retry {attempt + 1}/{self.max_retries}")
                self._close()
                time.sleep(min(2 ** attempt, 8))
        raise RuntimeError(f"Could not stream a pulse-train window after {self.max_retries} attempts") from last_err

    def _sample_once(self, span_us, rng) -> pd.DataFrame:
        ds, c = self._h5["data"], self._layout
        labels_ds = self._h5["labels"] if "labels" in self._h5 else None
        n = ds.shape[0]
        if n < 10:
            raise ValueError("pulse train too short")
        rows = self.initial_rows
        pos = int(rng.integers(0, max(1, n - rows)))
        blocks, lblocks = [], []
        while True:
            end = min(n, pos + rows)
            blocks.append(ds[pos:end])
            if labels_ds is not None:
                lblocks.append(np.asarray(labels_ds[pos:end]).reshape(-1))
            arr = blocks[0] if len(blocks) == 1 else np.concatenate(blocks)
            if arr[-1, c["toa"]] - arr[0, c["toa"]] >= span_us or end >= n:
                break
            pos, rows = end, min(rows * 2, self.max_rows_per_read)
        lab = (np.concatenate(lblocks) if lblocks else np.full(len(arr), -1)).astype(np.int64)
        toa = arr[:, c["toa"]].astype(np.float64)
        order = np.argsort(toa, kind="stable")
        arr, toa, lab = arr[order], toa[order], lab[order]
        toa = toa - toa[0]
        keep = toa < span_us
        arr, toa, lab = arr[keep], toa[keep], lab[keep]
        freq = arr[:, c["freq"]].astype(np.float64)
        scale = {"mhz": 1e-3, "ghz": 1.0, "hz": 1e-9}.get(self.freq_unit.lower())
        if scale is None:                                   # auto: > 100 means MHz
            scale = 1e-3 if np.nanpercentile(freq, 99) > 100.0 else 1.0
        aoa = arr[:, c["aoa"]].astype(np.float64)
        if self.aoa_unit == "rad":
            aoa = np.degrees(aoa)
        return pd.DataFrame({"toa_us": toa, "freq_ghz": freq * scale,
                             "pw_us": arr[:, c["pw"]].astype(np.float64), "aoa_deg": aoa,
                             "amplitude_db": arr[:, c["amp"]].astype(np.float64), "emitter_id": lab})

    def stats(self) -> str:
        fetched = self.transport_bytes + self._transport_now()
        return (f"{self.files_opened} file(s) opened; ~{fetched / 2 ** 20:.1f} MB fetched over the network "
                f"(h5py read {self.bytes_requested / 2 ** 20:.1f} MB); nothing saved to disk")


def make_hf_fs(token: Optional[str] = None):
    try:
        from huggingface_hub import HfFileSystem
    except ImportError as e:
        raise ImportError("pip install huggingface_hub") from e
    token = token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_TOKEN")
    if token is None:
        print("  [source] no HF token given; using the cached login if any. TSRD is GATED: accept its "
              "terms on the dataset page and pass --hf-token / set HF_TOKEN.")
    return HfFileSystem(token=token)


def make_source(kind: str, mode: str, subset: str, hf_token: Optional[str] = None,
                local_dir: Optional[str] = None, fs=None, **kw) -> PdwSource:
    if kind == "hf":
        return H5StreamSource(fs or make_hf_fs(hf_token), f"datasets/{TSRD_REPO}", mode, subset, **kw)
    if kind == "local":
        import fsspec
        if not local_dir:
            raise ValueError("--source local needs --local-dir (folder containing stare/ and scan/)")
        return H5StreamSource(fsspec.filesystem("file"), local_dir, mode, subset, **kw)
    raise ValueError(f"unknown source '{kind}'")


def add_data_args(ap: argparse.ArgumentParser, default_subset: str, default_mode: Optional[str]) -> None:
    """Data-source CLI flags shared by train_dqn.py and evaluate.py."""
    ap.add_argument("--source", choices=["hf", "local"], default="hf")
    ap.add_argument("--mode", choices=["stare", "scan"], default=default_mode)
    ap.add_argument("--subset", default=default_subset, help="train | validation | test")
    ap.add_argument("--local-dir", default=None)
    ap.add_argument("--hf-token", default=None)
    ap.add_argument("--block-size-mb", type=float, default=0.5)
    ap.add_argument("--cache-type", default="blockcache", choices=["blockcache", "readahead", "bytes"])
    ap.add_argument("--episodes-per-file", type=int, default=4)


def source_from_args(a, mode: str, subset: str, fs=None) -> PdwSource:
    if fs is None and a.source == "hf":
        fs = make_hf_fs(a.hf_token)
    return make_source(a.source, mode, subset, hf_token=a.hf_token, local_dir=a.local_dir, fs=fs,
                       block_size_mb=a.block_size_mb, cache_type=a.cache_type,
                       episodes_per_file=a.episodes_per_file)


def make_env(cfg: Config, source: PdwSource, seed: Optional[int] = None) -> "RfEnvironment":
    return RfEnvironment(source, cfg.num_bins, cfg.k_observed, cfg.snr_db, cfg.freq_range_ghz,
                         cfg.frame_window_us, cfg.episode_length, cfg.detection_threshold,
                         cfg.false_alarm_penalty, seed=seed)


class RfEnvironment:
    def __init__(self, source: PdwSource, num_bins=10, k_observed=3, snr_db=10.0,
                 freq_range_ghz=(2.0, 6.0), frame_window_us=10.0, episode_length=200,
                 detection_threshold=0.5, false_alarm_penalty=2.0,
                 min_inband_pulses: Optional[int] = None, max_resample=6, seed: Optional[int] = None):
        self.source = source
        self.num_bins, self.k_observed = num_bins, k_observed
        self.freq_range = tuple(freq_range_ghz)
        self.bin_width = (self.freq_range[1] - self.freq_range[0]) / num_bins
        self.frame_window_us, self.episode_length = frame_window_us, episode_length
        self.detection_threshold, self.false_alarm_penalty = detection_threshold, false_alarm_penalty
        self.min_inband = max(5, episode_length // 20) if min_inband_pulses is None else min_inband_pulses
        self.max_resample = max_resample
        self.rng = np.random.default_rng(seed)
        self.noise_rng = np.random.default_rng(None if seed is None else seed + 10_007)
        self.set_snr(snr_db)
        self.frames: List[List[dict]] = [[] for _ in range(episode_length)]
        self.current_step = 0
        self.last_episode_stats: Dict[str, float] = {}

    def set_snr(self, snr_db: float) -> None:
        self.snr_db = snr_db
        self.noise_std = math.sqrt(1.0 / (2.0 * 10.0 ** (snr_db / 10.0)))

    def _freq_to_bin(self, f):
        return np.clip(((np.asarray(f) - self.freq_range[0]) / self.bin_width).astype(int), 0, self.num_bins - 1)

    def _frames_from_df(self, df: pd.DataFrame) -> List[List[dict]]:
        lo, hi = self.freq_range
        inband = df[(df["freq_ghz"] >= lo) & (df["freq_ghz"] <= hi)]
        frame_idx = np.floor(inband["toa_us"].to_numpy() / self.frame_window_us).astype(int)
        self.last_episode_stats = {
            "raw_pulses": float(len(df)), "inband_pulses": float(len(inband)),
            "inband_per_frame": float(len(inband)) / self.episode_length,
            "emitters_inband": float(inband["emitter_id"].nunique()) if len(inband) else 0.0}
        frames: List[List[dict]] = [[] for _ in range(self.episode_length)]
        cols = {c: inband[c].to_numpy() for c in PDW_COLUMNS}
        bins = self._freq_to_bin(cols["freq_ghz"])
        for i in range(len(inband)):
            f = int(frame_idx[i])
            if 0 <= f < self.episode_length:
                frames[f].append({"emitter_id": int(cols["emitter_id"][i]), "toa_us": float(cols["toa_us"][i]),
                                  "freq_ghz": float(cols["freq_ghz"][i]), "pw_us": float(cols["pw_us"][i]),
                                  "aoa_deg": float(cols["aoa_deg"][i]),
                                  "amplitude_db": float(cols["amplitude_db"][i]), "bin_idx": int(bins[i])})
        return frames

    def fetch_episode(self) -> List[List[dict]]:
        span = self.episode_length * self.frame_window_us
        frames = None
        for _ in range(self.max_resample):
            frames = self._frames_from_df(self.source.sample_window(span, self.rng))
            if self.last_episode_stats["inband_pulses"] >= self.min_inband:
                break
        return frames

    def reset(self, frames: Optional[List[List[dict]]] = None, noise_seed: Optional[int] = None):
        """frames=None streams a fresh window; pass stored frames to replay identical data."""
        self.frames = frames if frames is not None else self.fetch_episode()
        if noise_seed is not None:
            self.noise_rng = np.random.default_rng(noise_seed)
        self.current_step = 0
        return {"dqn_power_vector": np.zeros(self.num_bins), "intercepted_pdws": []}

    def step(self, action_monitored_bins: Sequence[int]):
        bins = sorted({int(b) for b in action_monitored_bins})
        if len(bins) != self.k_observed or bins[0] < 0 or bins[-1] >= self.num_bins:
            raise ValueError(f"Action must be {self.k_observed} distinct bins in [0, {self.num_bins - 1}]")
        pdws = self.frames[self.current_step]
        N = self.num_bins

        counts = np.zeros(N)
        for p in pdws:
            counts[p["bin_idx"]] += 1
        active = counts > 0
        noisy = active.astype(float) + self.noise_rng.normal(0.0, self.noise_std, N)
        mask = np.zeros(N, dtype=bool)
        mask[bins] = True
        y = np.where(mask, noisy, 0.0)
        detected = mask & (y >= self.detection_threshold)

        intercepted = [p for p in pdws if detected[p["bin_idx"]]]
        false_alarms = int(np.sum(detected & ~active))
        empty_monitored = int(np.sum(mask & ~active))
        n_total, n_int = len(pdws), len(intercepted)
        n_missed = n_total - n_int

        reward = n_int * 10.0 - n_missed * 5.0 - false_alarms * self.false_alarm_penalty
        pd_val = n_int / n_total if n_total else 0.0
        pfa = false_alarms / empty_monitored if empty_monitored else 0.0

        self.current_step += 1
        done = self.current_step >= self.episode_length
        info = {"total_active_pulses": n_total, "missed_pulses": n_missed, "intercepted": n_int,
                "false_alarms": false_alarms, "empty_monitored": empty_monitored,
                "productive_slots": int(np.sum(mask & active)),
                "oracle_capture": float(np.sort(counts)[::-1][:self.k_observed].sum()),
                "active_bins": np.flatnonzero(active).tolist()}
        obs = {"dqn_power_vector": y, "intercepted_pdws": intercepted}
        return obs, reward, done, info, noisy, pd_val, pfa


def wrap(x):
    return x - np.floor(x + 0.5)


def _cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


class PulseEKF:
    def __init__(self, q_phase=1e-5, q_period=1e-5):
        self.q_phase = q_phase
        self.q_period = q_period
        self.x = np.array([0.0, 1.0])
        self.P = np.eye(2)
        self.t = 0.0

    def initialize(self, t0, period, meas_std, period_rel_std=0.05):
        self.x = np.array([0.0, float(period)])
        self.P = np.diag([(meas_std / period) ** 2, (period * period_rel_std) ** 2])
        self.t = t0

    def peek(self, t):
        dt = t - self.t
        T = self.x[1]
        x = np.array([wrap(self.x[0] + dt / T), T])
        F = np.array([[1.0, -dt / T ** 2], [0.0, 1.0]])
        Q = np.diag([self.q_phase * abs(dt), self.q_period * abs(dt)])
        return x, F @ self.P @ F.T + Q

    def predict(self, t):
        self.x, self.P = self.peek(t)
        self.t = t

    def innovation(self, t, meas_std):
        x, P = self.peek(t)
        R = (meas_std / x[1]) ** 2
        return -wrap(x[0]), P[0, 0] + R

    def update_toa(self, t, meas_std):
        self.predict(t)
        R = (meas_std / self.x[1]) ** 2
        H = np.array([[1.0, 0.0]])
        y = -wrap(self.x[0])
        S = (H @ self.P @ H.T).item() + R
        K = (self.P @ H.T) / S
        self.x = self.x + K.flatten() * y
        self.x[0] = wrap(self.x[0])
        I_KH = np.eye(2) - K @ H
        self.P = I_KH @ self.P @ I_KH.T + K @ K.T * R

    def relock(self, t, meas_std):
        self.predict(t)
        R = (meas_std / self.x[1]) ** 2
        self.x[0] = 0.0
        self.P[0, :] = 0.0
        self.P[:, 0] = 0.0
        self.P[0, 0] = R
        self.P[1, 1] = max(self.P[1, 1], (0.05 * self.x[1]) ** 2)

    def window_probability(self, t_a, t_b):
        x, P = self.peek(t_a)
        mu = wrap(x[0])
        s = math.sqrt(max(P[0, 0], 1e-6))
        delta = (t_b - t_a) / x[1]
        p = 0.0
        for k in (-1, 0, 1):
            p += _cdf((0.0 - (mu + k)) / s) - _cdf((-delta - (mu + k)) / s)
        return float(min(max(p, 0.0), 1.0))

    def next_pulse_time(self, t):
        x, _ = self.peek(t)
        dt = ((-x[0]) % 1.0) * x[1]
        return t + (dt if dt > 1e-9 else x[1])

    @property
    def phase_std(self):
        return math.sqrt(max(self.P[0, 0], 0.0))

    @property
    def period(self):
        return self.x[1]


def estimate_pri(toas, pri_min, pri_max, tol=0.7, min_cycles=2):
    d = np.asarray(toas, float) - toas[0]
    cands = set()
    for i in range(len(d)):
        for j in range(i + 1, len(d)):
            gap = d[j] - d[i]
            for n in range(1, int(gap / pri_min) + 1):
                T = gap / n
                if pri_min <= T <= pri_max:
                    cands.add(round(T, 3))
    for T in sorted(cands, reverse=True):
        n = np.round(d / T)
        if len(set(n.tolist())) < len(n):
            continue
        A = np.vstack([np.ones_like(n), n]).T
        (t0, Tls), *_ = np.linalg.lstsq(A, d, rcond=None)
        resid = np.max(np.abs(d - (t0 + n * Tls)))
        if pri_min <= Tls <= pri_max and resid <= tol and (n.max() - n.min()) >= min_cycles:
            return float(Tls), float(resid)
    return None


class _Track:
    def __init__(self, tid, bin_idx, aoa, k):
        self.id, self.bin, self.aoa = tid, bin_idx, aoa
        self.toas = []
        self.ekf = None
        self.confirmed = False
        self.last_hit = k
        self.miss = 0
        self.n_hits = 0
        self.verify_left = 0
        self.verified = False
        self.outliers = 0


class EKFTrackBank:
    def __init__(self, num_bins, pri_min=2.0, pri_max=30.0, meas_std=0.29, aoa_gate_deg=5.0,
                 confirm_hits=5, tentative_timeout=None, miss_limit=4, gate_sigmas=4.0,
                 lost_std=0.15, q_phase=1e-5, q_period=1e-5, max_toas=10):
        self.num_bins = num_bins
        self.pri_min, self.pri_max = pri_min, pri_max
        self.meas_std = meas_std
        self.aoa_gate = aoa_gate_deg
        self.confirm_hits = confirm_hits
        self.timeout = tentative_timeout or 3 * pri_max
        self.miss_limit = miss_limit
        self.gate_sigmas, self.lost_std = gate_sigmas, lost_std
        self.q_phase, self.q_period = q_phase, q_period
        self.max_toas = max_toas
        self.ekf_dim = 4 * num_bins
        self.reset()

    def reset(self):
        self.k = 0
        self.tracks = []
        self._next_id = 0
        self.last_monitored = np.full(self.num_bins, -1)

    def _associate(self, pdw):
        best, best_d = None, self.aoa_gate
        for tr in self.tracks:
            if tr.bin == pdw["bin_idx"]:
                d = abs((pdw.get("aoa_deg", 0.0) - tr.aoa + 180) % 360 - 180)
                if d <= best_d:
                    best, best_d = tr, d
        return best

    def _build(self, tr, T):
        ekf = PulseEKF(self.q_phase, self.q_period)
        ekf.initialize(tr.toas[0], T, self.meas_std)
        for t in tr.toas[1:]:
            ekf.update_toa(t, self.meas_std)
        tr.ekf, tr.confirmed, tr.miss, tr.outliers = ekf, True, 0, 0
        tr.verified, tr.verify_left = False, int(math.ceil(T)) + 2

    def _confirm(self, tr):
        if len(tr.toas) < self.confirm_hits:
            return
        est = estimate_pri(tr.toas, self.pri_min, self.pri_max)
        if est is not None:
            self._build(tr, est[0])

    def _reestimate(self, tr):
        if len(tr.toas) < 3:
            return False
        est = estimate_pri(tr.toas, self.pri_min, self.pri_max)
        if est is None:
            return False
        self._build(tr, est[0])
        return True

    def update(self, monitored_bins, obs):
        k, t_meas = self.k, self.k + 0.5
        monitored = set(int(b) for b in monitored_bins)
        expected = {tr.id: tr.ekf.window_probability(k, k + 1) for tr in self.tracks if tr.confirmed}
        hit_ids = set()

        for pdw in obs.get("intercepted_pdws", []):
            tr = self._associate(pdw)
            if tr is None:
                tr = _Track(self._next_id, pdw["bin_idx"], pdw.get("aoa_deg", 0.0), k)
                self._next_id += 1
                self.tracks.append(tr)
            if tr.id in hit_ids:
                continue
            aoa = pdw.get("aoa_deg", tr.aoa)
            tr.aoa += 0.1 * (aoa - tr.aoa)
            tr.toas = (tr.toas + [t_meas])[-self.max_toas:]
            tr.last_hit, tr.n_hits = k, tr.n_hits + 1
            hit_ids.add(tr.id)
            if tr.confirmed:
                y, S = tr.ekf.innovation(t_meas, self.meas_std)
                if abs(y) <= self.gate_sigmas * math.sqrt(S):
                    tr.ekf.update_toa(t_meas, self.meas_std)
                    tr.miss, tr.outliers = 0, max(0, tr.outliers - 1)
                elif self._reestimate(tr):
                    pass
                elif tr.ekf.phase_std > self.lost_std:
                    tr.ekf.relock(t_meas, self.meas_std)
                    tr.miss = 0
                else:
                    tr.outliers += 1
            else:
                self._confirm(tr)

        for tr in self.tracks:
            if tr.confirmed and tr.bin in monitored:
                if tr.id not in hit_ids and expected[tr.id] > 0.5:
                    tr.miss += 1
                if tr.verify_left > 0:
                    tr.verify_left -= 1
                    tr.verified = tr.verify_left == 0
        self.tracks = [tr for tr in self.tracks
                       if (tr.confirmed and tr.miss < (self.miss_limit if tr.verified else 2) and tr.outliers < 6)
                       or (not tr.confirmed and k - tr.last_hit <= self.timeout)]
        for b in monitored:
            self.last_monitored[b] = k
        self.k += 1

    def bin_probabilities(self):
        p = np.zeros(self.num_bins)
        for tr in self.tracks:
            if tr.confirmed:
                p[tr.bin] = max(p[tr.bin], tr.ekf.window_probability(self.k, self.k + 1))
        return p

    def features(self):
        f = np.zeros((self.num_bins, 4), dtype=np.float32)
        best = np.full(self.num_bins, -1.0)
        for tr in self.tracks:
            if not tr.confirmed:
                continue
            p = tr.ekf.window_probability(self.k, self.k + 1)
            if p > best[tr.bin]:
                best[tr.bin] = p
                tnext = tr.ekf.next_pulse_time(self.k) - self.k
                f[tr.bin] = [p, min(tnext / self.pri_max, 1.0), min(tr.ekf.phase_std / 0.5, 1.0),
                             1.0 if tr.verified else 0.5]
        return f.reshape(-1)


def build_action_table(num_bins: int, k_observed: int) -> List[Tuple[int, ...]]:
    return list(itertools.combinations(range(num_bins), k_observed))


class TemporalAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh(), nn.Linear(hidden_size, 1))

    def forward(self, x):
        weights = torch.softmax(self.score(x).squeeze(-1), dim=1)
        return torch.sum(x * weights.unsqueeze(-1), dim=1), weights


class GRU_FeatureExtractor(nn.Module):
    def __init__(self, num_bins, hidden_dim=64, num_layers=2, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_layer = nn.Sequential(nn.Linear(num_bins * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.gru = nn.GRU(hidden_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.attention = TemporalAttention(hidden_dim)

    def forward(self, sequence_history):
        seq, _ = self.gru(self.input_layer(sequence_history))
        context, _ = self.attention(seq)
        return context


class FrameEncoder(nn.Module):
    """Non-recurrent control: sees only the latest frame."""

    def __init__(self, num_bins, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.net = nn.Sequential(nn.Linear(num_bins * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())

    def forward(self, sequence_history):
        return self.net(sequence_history[:, -1, :])


class IntegratedAgentNetwork(nn.Module):
    """extractor -> [+ EKF features] -> dueling head:  Q = V + A - mean(A)."""

    def __init__(self, num_bins, num_actions, recurrent=True, ekf_dim=0, hidden_dim=64,
                 head_dim=128, gru_dropout=0.0):
        super().__init__()
        self.extractor = (GRU_FeatureExtractor(num_bins, hidden_dim, dropout=gru_dropout) if recurrent
                          else FrameEncoder(num_bins, hidden_dim))
        self.ekf_dim = ekf_dim
        self.trunk = nn.Sequential(nn.Linear(hidden_dim + ekf_dim, head_dim), nn.ReLU())
        self.value_head = nn.Linear(head_dim, 1)
        self.advantage_head = nn.Linear(head_dim, num_actions)

    def forward(self, sequence_history, ekf_data=None):
        z = self.extractor(sequence_history)
        if self.ekf_dim > 0:
            z = torch.cat([z, ekf_data], dim=1)
        h = self.trunk(z)
        adv = self.advantage_head(h)
        return self.value_head(h) + adv - adv.mean(dim=1, keepdim=True)


def build_network(cfg: Config, num_actions: int, recurrent: bool) -> IntegratedAgentNetwork:
    return IntegratedAgentNetwork(cfg.num_bins, num_actions, recurrent,
                                  4 * cfg.num_bins if cfg.use_ekf else 0,
                                  cfg.hidden_dim, cfg.head_dim, cfg.gru_dropout)

class StateBuilder:
    """Sliding window of (power vector, monitored mask) frames + optional EKF features."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.N, self.L = cfg.num_bins, cfg.seq_len
        self.bank = (EKFTrackBank(cfg.num_bins, pri_min=cfg.ekf_pri_min, pri_max=cfg.ekf_pri_max)
                     if cfg.use_ekf else None)
        self.reset()

    def reset(self):
        self.history = deque([np.zeros(2 * self.N, np.float32) for _ in range(self.L)], maxlen=self.L)
        self.steps_since = np.zeros(self.N)
        if self.bank is not None:
            self.bank.reset()

    def _frame(self, obs, bins):
        mask = np.zeros(self.N, np.float32)
        mask[list(bins)] = 1.0
        return np.concatenate([np.asarray(obs["dqn_power_vector"], np.float32), mask])

    def observe(self, monitored_bins, obs):
        self.history.append(self._frame(obs, monitored_bins))
        self.steps_since += 1
        self.steps_since[list(monitored_bins)] = 0
        if self.bank is not None:
            self.bank.update(monitored_bins, obs)

    def state(self) -> Tuple[np.ndarray, np.ndarray]:
        seq = np.stack(self.history).astype(np.float32)
        ekf = self.bank.features().astype(np.float32) if self.bank is not None else np.zeros(0, np.float32)
        return seq, ekf

    def stalest_bins(self, k: int, rng) -> List[int]:
        return sorted(np.argsort(-(self.steps_since + rng.random(self.N) * 1e-3))[:k].tolist())


class RunningStats:
    def __init__(self):
        self.steps = self.pulses = self.intercepted = self.false_alarms = 0
        self.empty_slots = self.productive = self.slots = 0
        self.oracle = self.base_reward = self.shaped_reward = 0.0

    def add(self, info, k, base_reward, shaped_reward=0.0):
        self.steps += 1
        self.pulses += info["total_active_pulses"]
        self.intercepted += info["intercepted"]
        self.false_alarms += info["false_alarms"]
        self.empty_slots += info["empty_monitored"]
        self.productive += info["productive_slots"]
        self.slots += k
        self.oracle += info["oracle_capture"]
        self.base_reward += base_reward
        self.shaped_reward += shaped_reward

    def summary(self) -> Dict[str, float]:
        nan = float("nan")
        return {"pd": self.intercepted / self.pulses if self.pulses else nan,
                "pfa": self.false_alarms / self.empty_slots if self.empty_slots else nan,
                "scan_eff": self.intercepted / self.oracle if self.oracle else nan,
                "productive_frac": self.productive / self.slots if self.slots else nan,
                "base_reward": self.base_reward, "steps": self.steps, "pulses": self.pulses}


def composite_score(m: Dict[str, float], pfa_budget: float = 0.05) -> float:
    nz = lambda x: 0.0 if math.isnan(x) else x
    return 0.5 * nz(m["pd"]) + 0.5 * nz(m["scan_eff"]) - 5.0 * max(0.0, nz(m["pfa"]) - pfa_budget)

def freeze_network(net: nn.Module, cfg: Config, path: str, recurrent: bool, double: bool,
                   num_actions: int, extra: Optional[Dict] = None) -> str:
    meta = asdict(cfg)
    meta["freq_range_ghz"] = list(cfg.freq_range_ghz)
    meta.update({"recurrent": recurrent, "double": double, "num_actions": num_actions})
    if extra:
        meta.update(extra)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    torch.save({"state_dict": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "meta": meta}, path)
    with open(path + ".json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  frozen checkpoint: {path}  (+ {os.path.basename(path)}.json)")
    return path


class FrozenScheduler:
    """Inference-only DQN scheduler.  Same interface as the baseline scanners:
        reset() / select_bins() / observe(bins, obs)"""

    def __init__(self, net: IntegratedAgentNetwork, cfg: Config, meta: Dict, device):
        self.net, self.cfg, self.meta, self.device = net, cfg, meta, device
        self.actions = build_action_table(cfg.num_bins, cfg.k_observed)
        self.builder = StateBuilder(cfg)

    def reset(self):
        self.builder.reset()

    def select_bins(self) -> List[int]:
        seq, ekf = self.builder.state()
        with torch.no_grad():
            q = self.net(torch.as_tensor(seq, device=self.device).unsqueeze(0),
                         torch.as_tensor(ekf, device=self.device).unsqueeze(0) if ekf.size else None)
        return list(self.actions[int(q.argmax(1).item())])

    def observe(self, monitored_bins, obs):
        self.builder.observe(monitored_bins, obs)


def load_frozen_scheduler(path: str, device: Optional[torch.device] = None) -> FrozenScheduler:
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(path, map_location=device, weights_only=True)
    meta = ckpt["meta"]
    fields = Config.__dataclass_fields__
    cfg = Config(**{k: (tuple(v) if k == "freq_range_ghz" else v) for k, v in meta.items() if k in fields})
    net = build_network(cfg, meta["num_actions"], meta["recurrent"]).to(device)
    net.load_state_dict(ckpt["state_dict"])
    net.eval()
    for p in net.parameters():
        p.requires_grad_(False)
    return FrozenScheduler(net, cfg, meta, device)

def probe_source(source: PdwSource, cfg: Config, n_windows: int = 3) -> None:
    env = make_env(cfg, source, seed=0)
    print(f"\n=== Probing the stream ({n_windows} windows of {cfg.episode_length * cfg.frame_window_us:g} us) ===")
    s = {"inband_per_frame": 1.0}
    for i in range(n_windows):
        t0 = time.time()
        env.reset()
        s = env.last_episode_stats
        occ = np.bincount([p["bin_idx"] for fr in env.frames for p in fr], minlength=cfg.num_bins)
        print(f"  window {i}: {s['raw_pulses']:.0f} pulses, {s['inband_pulses']:.0f} in "
              f"{cfg.freq_range_ghz[0]:g}-{cfg.freq_range_ghz[1]:g} GHz ({s['inband_per_frame']:.2f}/frame, "
              f"{s['emitters_inband']:.0f} emitters) | per-bin {occ.tolist()} | {time.time() - t0:.1f}s")
    print(f"  {source.stats()}")
    if s["inband_per_frame"] < 0.1:
        print("  WARNING: very sparse in-band traffic. Increase --frame-window-us (50-100) or widen --freq-range.")


def write_mock_tsrd(root: str, mode: str = "stare", subset: str = "train", n_files: int = 3,
                    n_pulses: int = 300_000, seed: int = 0) -> str:
    """Writes TSRD-format .h5 files so the whole streaming path can be tested offline."""
    import h5py
    rng = np.random.default_rng(seed)
    d = os.path.join(root, mode, f"{subset}_{mode}")
    os.makedirs(d, exist_ok=True)
    for f_i in range(n_files):
        toa, cf, lab = [], [], []
        span_us = n_pulses * 12.0
        for e in range(int(rng.integers(6, 12))):
            pri = rng.uniform(40, 400)
            times = np.arange(rng.uniform(0, pri), span_us, pri)[: n_pulses // 4]
            times = times + rng.normal(0, 0.3, len(times))
            if e % 3 == 0:
                f = rng.uniform(1000, 12000, len(times))
            else:
                f = np.full(len(times), rng.uniform(1500, 9000)) + rng.normal(0, 3, len(times))
            toa.append(times); cf.append(f); lab.append(np.full(len(times), e))
        toa, cf, lab = np.concatenate(toa), np.concatenate(cf), np.concatenate(lab)
        o = np.argsort(toa)
        toa, cf, lab = toa[o], cf[o], lab[o]
        data = np.stack([toa, cf, rng.uniform(1, 20, len(toa)), rng.uniform(0, 360, len(toa)),
                         rng.uniform(-60, -30, len(toa))], axis=1).astype(np.float32)
        with h5py.File(os.path.join(d, f"pulse_train_{f_i}.h5"), "w") as h:
            h.create_dataset("data", data=data, dtype=np.float32, compression="gzip", compression_opts=4,
                             chunks=(8192, 5))
            h.create_dataset("labels", data=lab.astype(np.int8), compression="gzip")
            h.create_group("metadata").create_dataset(
                "feature_names", data=np.array(["ToA", "CF", "PW", "AoA", "Amp"], dtype="S"))
    return root